In [1]:
import numpy as np
import pandas as pd
import os
import tokenizers
import string
import torch
import transformers
import torch.nn as nn
from torch.nn import functional as F
from tqdm import tqdm
import re
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoConfig, AutoModel, PreTrainedModel
from torch.optim.swa_utils import AveragedModel, SWALR
from torch.utils.data import Sampler

seed everything we have

In [2]:
MAX_LEN = 192
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 8
MODEL_NAME = "/kaggle/input/models/cbdsigmas/deberta-v3-large/pytorch/default/1/deberta-v3-large"
TOKENIZER = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)
BASE_SEED = 42
EPOCHS = 4

In [3]:
def seed_everything(seed):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
class TweetModel(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.roberta = transformers.AutoModel.from_pretrained(
            MODEL_NAME,
            config=conf,
            attn_implementation="eager",
            torch_dtype=torch.float32
        )

        hidden = conf.hidden_size

        self.layer_norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, 2)

        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, ids, mask, token_type_ids=None):
        # DeBERTa-v3 does not use token_type_ids; avoid passing them.
        outputs = self.roberta(
            input_ids=ids,
            attention_mask=mask,
            return_dict=False
        )

        sequence_output = outputs[0]

        # Safety: avoid propagating non-finite backbone activations.
        if not torch.isfinite(sequence_output).all():
            sequence_output = torch.nan_to_num(sequence_output, nan=0.0, posinf=1e4, neginf=-1e4)

        with torch.cuda.amp.autocast(enabled=False):
            x = self.layer_norm(sequence_output.float())
            x = self.dropout(x)
            logits = self.classifier(x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)

        start_logits, end_logits = logits.split(1, dim=-1)
        return start_logits.squeeze(-1), end_logits.squeeze(-1)


In [5]:
def pp(filtered_output, real_tweet):
    filtered_output = ' '.join(filtered_output.split())
    if len(real_tweet.split()) < 2:
        filtered_output = real_tweet
    else:
        if len(filtered_output.split()) == 1:
            if filtered_output.endswith(".."):
                if real_tweet.startswith(" "):
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\.){2,}', '', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\.){2,}', '.', filtered_output)
                else:
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\.){2,}', '.', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\.){2,}', '..', filtered_output)
                return filtered_output
            if filtered_output.endswith('!!'):
                if real_tweet.startswith(" "):
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\!){2,}', '', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\!){2,}', '!', filtered_output)
                else:
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\!){2,}', '!', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\!){2,}', '!!', filtered_output)
                return filtered_output

        if real_tweet.startswith(" "):
            filtered_output = filtered_output.strip()
            text_annotetor = ' '.join(real_tweet.split())
            start = text_annotetor.find(filtered_output)
            end = start + len(filtered_output)
            start -= 0
            end += 2
            flag = real_tweet.find("  ")
            if flag < start:
                filtered_output = real_tweet[start:end]

        if "  " in real_tweet and not real_tweet.startswith(" "):
            filtered_output = filtered_output.strip()
            text_annotetor = re.sub(" {2,}", " ", real_tweet)
            start = text_annotetor.find(filtered_output)
            end = start + len(filtered_output)
            start -= 0
            end += 2
            flag = real_tweet.find("  ")
            if flag < start:
                filtered_output = real_tweet[start:end]
    return filtered_output

def process_data(tweet, selected_text, sentiment, tokenizer, max_len):
    tweet = " " + " ".join(str(tweet).split())
    selected_text = " " + " ".join(str(selected_text).split())

    # locate selected text
    len_st = len(selected_text) - 1
    idx0, idx1 = None, None

    for ind in (i for i, e in enumerate(tweet) if e == selected_text[1]):
        if " " + tweet[ind: ind+len_st] == selected_text:
            idx0 = ind
            idx1 = ind + len_st - 1
            break

    char_targets = [0] * len(tweet)
    if idx0 is not None and idx1 is not None:
        for ct in range(idx0, idx1 + 1):
            char_targets[ct] = 1

    # reserve 5 tokens: [CLS] sentiment [SEP] [SEP] ... [SEP]
    max_tweet_tokens = max_len - 5

    # encode tweet WITH truncation to keep shapes valid
    tweet_enc = tokenizer(
        tweet,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_tweet_tokens
    )

    input_ids_orig = tweet_enc["input_ids"]
    tweet_offsets = tweet_enc["offset_mapping"]

    # encode sentiment token
    sentiment_enc = tokenizer(
        sentiment,
        add_special_tokens=False
    )
    sentiment_id = sentiment_enc["input_ids"][0]

    # build final sequence (same as Kaggle)
    input_ids = (
        [tokenizer.cls_token_id] +
        [sentiment_id] +
        [tokenizer.sep_token_id] +
        [tokenizer.sep_token_id] +
        input_ids_orig +
        [tokenizer.sep_token_id]
    )

    offsets = (
        [(0,0)] * 4 +
        tweet_offsets +
        [(0,0)]
    )

    # find target indices
    target_idx = []
    for j, (o1, o2) in enumerate(tweet_offsets):
        if sum(char_targets[o1:o2]) > 0:
            target_idx.append(j)

    if len(target_idx) > 0:
        targets_start = target_idx[0] + 4
        targets_end = target_idx[-1] + 4
    else:
        targets_start = 0
        targets_end = 0

    # clamp in case selected span was truncated
    max_idx = len(input_ids) - 1
    targets_start = min(targets_start, max_idx)
    targets_end = min(targets_end, max_idx)

    # defensive truncate (should already be <= max_len)
    input_ids = input_ids[:max_len]
    offsets = offsets[:max_len]

    # padding
    padding_length = max_len - len(input_ids)
    mask = [1] * len(input_ids) + [0] * padding_length
    input_ids = input_ids + [tokenizer.pad_token_id] * padding_length
    token_type_ids = [0] * max_len
    offsets = offsets + [(0,0)] * padding_length

    return {
        "ids": input_ids,
        "mask": mask,
        "token_type_ids": token_type_ids,
        "targets_start": targets_start,
        "targets_end": targets_end,
        "orig_tweet": tweet,
        "orig_selected": selected_text,
        "sentiment": sentiment,
        "offsets": offsets
    }


class TweetDataset:
    def __init__(self, tweet, sentiment, selected_text):
        self.tweet = tweet
        self.sentiment = sentiment
        self.selected_text = selected_text
        self.tokenizer = TOKENIZER
        self.max_len = MAX_LEN
    
    def __len__(self):
        return len(self.tweet)

    def __getitem__(self, item):
        data = process_data(
            self.tweet[item], 
            self.selected_text[item], 
            self.sentiment[item],
            self.tokenizer,
            self.max_len
        )

        return {
            'ids': torch.tensor(data["ids"], dtype=torch.long),
            'mask': torch.tensor(data["mask"], dtype=torch.long),
            'token_type_ids': torch.tensor(data["token_type_ids"], dtype=torch.long),
            'targets_start': torch.tensor(data["targets_start"], dtype=torch.long),
            'targets_end': torch.tensor(data["targets_end"], dtype=torch.long),
            'orig_tweet': data["orig_tweet"],
            'orig_selected': data["orig_selected"],
            'sentiment': data["sentiment"],
            'offsets': torch.tensor(data["offsets"], dtype=torch.long)
        }


Added Sentiment Sampler

In [6]:
class SentimentSampler(Sampler):
    def __init__(self, sentiments, batch_size):
        self.batch_size = batch_size
        
        self.pos = np.where(sentiments == "positive")[0]
        self.neg = np.where(sentiments == "negative")[0]
        self.neu = np.where(sentiments == "neutral")[0]

        self.k = batch_size // 3
        self.n_batches = min(len(self.pos), len(self.neg), len(self.neu)) // self.k

    def __iter__(self):
        pos = np.random.permutation(self.pos)
        neg = np.random.permutation(self.neg)
        neu = np.random.permutation(self.neu)

        for i in range(self.n_batches):
            batch = np.concatenate([
                pos[i*self.k:(i+1)*self.k],
                neg[i*self.k:(i+1)*self.k],
                neu[i*self.k:(i+1)*self.k],
            ])
            np.random.shuffle(batch)
            yield batch.tolist()   # ← MUST be list

    def __len__(self):
        return self.n_batches

In [7]:
def jaccard(str1, str2):
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    c = a.intersection(b)
    return float(len(c)) / (len(a) + len(b) - len(c))

def calculate_jaccard_score(
    original_tweet, 
    target_string, 
    sentiment_val, 
    idx_start, 
    idx_end, 
    offsets
):
    if idx_end < idx_start:
        idx_end = idx_start
    
    filtered_output = ""
    for ix in range(idx_start, idx_end + 1):
        filtered_output += original_tweet[offsets[ix][0]: offsets[ix][1]]
        if ix+1 < len(offsets) and offsets[ix][1] < offsets[ix+1][0]:
            filtered_output += " "

    if sentiment_val == "neutral" or len(original_tweet.split()) < 2:
        filtered_output = original_tweet

    jac = jaccard(target_string.strip(), filtered_output.strip())
    return jac, filtered_output

In [8]:
df_test = pd.read_csv("../input/tweet-sentiment-extraction/test.csv")
df_test["text"] = df_test["text"].fillna("").astype(str)
df_test["sentiment"] = df_test["sentiment"].fillna("neutral").astype(str)
df_test.loc[:, "selected_text"] = df_test["text"].values


training ig

In [9]:
df = pd.read_csv("../input/tweet-sentiment-extraction/train.csv")

# Drop rows with missing values
df = df.dropna(subset=["text", "selected_text", "sentiment"])

# Rename to match dataset class
df = df.rename(columns={
    "text": "tweet"
})

# Normalize whitespace (important for span alignment)
df["tweet"] = df["tweet"].astype(str).apply(
    lambda x: " ".join(x.split())
)
df["selected_text"] = df["selected_text"].astype(str).apply(
    lambda x: " ".join(x.split())
)

# Reset index (important for fold split)
df = df.reset_index(drop=True)

print("Train size:", len(df))
df.head()

Train size: 27480


,textID,tweet,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on th...","Sons of ****,",negative


In [10]:
device = torch.device("cuda")

model_config = AutoConfig.from_pretrained(MODEL_NAME)
model_config.output_hidden_states = True

loss function: CE of start + end pos

In [11]:
def loss_fn(start_logits, end_logits, start_positions, end_positions):
    seq_len = start_logits.size(1)

    valid = (
        (start_positions >= 0) &
        (end_positions >= 0) &
        (start_positions < seq_len) &
        (end_positions < seq_len)
    )

    if valid.sum() == 0:
        return torch.tensor(0.0, device=start_logits.device, requires_grad=True)

    start_logits = start_logits[valid]
    end_logits   = end_logits[valid]
    start_positions = start_positions[valid]
    end_positions   = end_positions[valid]

    loss_fct = nn.CrossEntropyLoss()
    start_loss = loss_fct(start_logits, start_positions)
    end_loss   = loss_fct(end_logits, end_positions)

    return (start_loss + end_loss) / 2

In [12]:
def train_fn(data_loader, model, optimizer, device, fold=None, epoch=None):
    model.train()
    total_loss = 0.0
    num_updates = 0
    
    pbar = tqdm(
        data_loader,
        desc=f"Fold {fold} - Epoch {epoch} [TRAIN]",
        leave=False
    )

    for batch in pbar:
        ids = batch["ids"].to(device)
        mask = batch["mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        targets_start = batch["targets_start"].to(device)
        targets_end = batch["targets_end"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs_start, outputs_end = model(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )

        outputs_start = outputs_start.float()
        outputs_end = outputs_end.float()

        if not torch.isfinite(outputs_start).all() or not torch.isfinite(outputs_end).all():
            print("⚠️ MODEL OUTPUT non-finite — skipping batch")
            continue

        loss = loss_fn(outputs_start, outputs_end, targets_start, targets_end)

        if not torch.isfinite(loss):
            print("⚠️ LOSS non-finite — skipping batch")
            continue

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if not torch.isfinite(grad_norm):
            print("⚠️ GRAD non-finite — skipping optimizer step")
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.step()

        total_loss += loss.item()
        num_updates += 1
        pbar.set_postfix(loss=loss.item())

    if num_updates == 0:
        return float("inf")

    return total_loss / num_updates


In [13]:
def eval_fn(data_loader, model, device, fold=None, epoch=None):
    model.eval()
    total_jaccard = 0
    
    pbar = tqdm(
        data_loader,
        desc=f"Fold {fold} - Epoch {epoch} [VAL]",
        leave=False
    )

    with torch.no_grad():
        for batch in pbar:
            ids = batch["ids"].to(device)
            mask = batch["mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)

            tweet = batch["orig_tweet"]
            selected = batch["orig_selected"]
            sentiment = batch["sentiment"]
            offsets = batch["offsets"].cpu().numpy()

            outputs_start, outputs_end = model(
                ids=ids,
                mask=mask,
                token_type_ids=token_type_ids
            )

            start_preds = torch.argmax(outputs_start, dim=1).cpu().numpy()
            end_preds = torch.argmax(outputs_end, dim=1).cpu().numpy()

            batch_jaccard = 0

            for i in range(len(tweet)):
                jac, pred_text = calculate_jaccard_score(
                    tweet[i],
                    selected[i],
                    sentiment[i],
                    start_preds[i],
                    end_preds[i],
                    offsets[i]
                )
                
                pred_text = pp(pred_text, tweet[i])
                jac = jaccard(selected[i], pred_text)
                
                batch_jaccard += jac

            batch_jaccard /= len(tweet)
            total_jaccard += batch_jaccard * len(tweet)

            pbar.set_postfix(jaccard=batch_jaccard)

    return total_jaccard / len(data_loader.dataset)

5-fold training

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(
        skf.split(df, df.sentiment)):

    seed_everything(BASE_SEED + 100*fold) # 100 because this is Kaggle trick

    print(f"FOLD {fold}")

    train_dataset = TweetDataset(
        df.tweet.values[train_idx],
        df.sentiment.values[train_idx],
        df.selected_text.values[train_idx]
    )

    val_dataset = TweetDataset(
        df.tweet.values[val_idx],
        df.sentiment.values[val_idx],
        df.selected_text.values[val_idx]
    )

    sampler = SentimentSampler(df.sentiment.values[train_idx], batch_size=8)
    train_loader = DataLoader(train_dataset, batch_sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=8)

    model = TweetModel(conf=model_config)
    model.to(device)

    # Lower LR + eps stabilizes DeBERTa-v3-large fine-tuning.
    optimizer = torch.optim.AdamW(model.parameters(), lr=7.5e-6, eps=1e-6, betas=(0.9, 0.999))

    best_jaccard = 0

    swa_model = AveragedModel(model)
    swa_start = int(4 * 0.75)
    
    best_jaccard = 0
    
    for epoch in range(4):
        train_loss = train_fn(train_loader, model, optimizer, device, fold, epoch)
        val_jaccard = eval_fn(val_loader, model, device, fold, epoch)
    
        if epoch >= swa_start:
            swa_model.update_parameters(model)
    
            # refresh BN before SWA eval
            torch.optim.swa_utils.update_bn(train_loader, swa_model)
    
            score_to_compare = eval_fn(val_loader, swa_model, device, fold, epoch)
            print(f"SWA Jaccard: {score_to_compare:.6f}")
            model_to_save = swa_model
        else:
            score_to_compare = val_jaccard
            model_to_save = model
    
        print(epoch, train_loss, val_jaccard)
    
        if score_to_compare > best_jaccard:
            best_jaccard = score_to_compare
            torch.save(model_to_save.state_dict(), f"model_{fold}.bin")

    torch.optim.swa_utils.update_bn(train_loader, swa_model)


FOLD 0


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

/tmp/ipykernel_55/2564028305.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Fold 0 - Epoch 0 [TRAIN]:   0%|          | 0/3112 [00:00<?, ?it/s]/tmp/ipykernel_55/3136268634.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
Fold 0 - Epoch 0 [TRAIN]:   0%|          | 4/3112 [00:02<19:53,  2.61it/s, loss=5.76]  

⚠️ MODEL OUTPUT NaN — skipping batch
⚠️ MODEL OUTPUT NaN — skipping batch
⚠️ MODEL OUTPUT NaN — skipping batch


Fold 0 - Epoch 0 [TRAIN]:   0%|          | 6/3112 [00:02<12:10,  4.25it/s, loss=5.76]

...

Fold 0 - Epoch 0 [TRAIN]:  39%|███▉      | 1210/3112 [01:28<02:23, 13.25it/s, loss=5.76]

⚠️ MODEL OUTPUT NaN — skipping batch
⚠️ MODEL OUTPUT NaN — skipping batch
⚠️ MODEL OUTPUT NaN — skipping batch


Fold 0 - Epoch 0 [TRAIN]:  39%|███▉      | 1212/3112 [01:28<02:23, 13.24it/s, loss=5.76]

⚠️ MODEL OUTPUT NaN — skipping batch
⚠️ MODEL OUTPUT NaN — skipping batch
⚠️ MODEL OUTPUT NaN — skipping batch


Fold 0 - Epoch 0 [TRAIN]:  39%|███▉      | 1214/3112 [01:28<02:23, 13.24it/s, loss=5.76]

In [ ]:
model1 = TweetModel(conf=model_config)
model1.to(device)
model1.load_state_dict(torch.load("model_0.bin"))
model1.eval()

model2 = TweetModel(conf=model_config)
model2.to(device)
model2.load_state_dict(torch.load("model_1.bin"))
model2.eval()

model3 = TweetModel(conf=model_config)
model3.to(device)
model3.load_state_dict(torch.load("model_2.bin"))
model3.eval()

model4 = TweetModel(conf=model_config)
model4.to(device)
model4.load_state_dict(torch.load("model_3.bin"))
model4.eval()

model5 = TweetModel(conf=model_config)
model5.to(device)
model5.load_state_dict(torch.load("model_4.bin"))
model5.eval()

In [ ]:
final_output = []

In [ ]:
test_dataset = TweetDataset(
        tweet=df_test.text.values,
        sentiment=df_test.sentiment.values,
        selected_text=df_test.selected_text.values
    )

data_loader = torch.utils.data.DataLoader(
    test_dataset,
    shuffle=False,
    batch_size=VALID_BATCH_SIZE,
    num_workers=1
)


with torch.no_grad():
    tk0 = tqdm(data_loader, total=len(data_loader))
    for bi, d in enumerate(tk0):
        ids = d["ids"]
        token_type_ids = d["token_type_ids"]
        mask = d["mask"]
        sentiment = d["sentiment"]
        orig_selected = d["orig_selected"]
        orig_tweet = d["orig_tweet"]
        targets_start = d["targets_start"]
        targets_end = d["targets_end"]
        offsets = d["offsets"].numpy()

        ids = ids.to(device, dtype=torch.long)
        token_type_ids = token_type_ids.to(device, dtype=torch.long)
        mask = mask.to(device, dtype=torch.long)
        targets_start = targets_start.to(device, dtype=torch.long)
        targets_end = targets_end.to(device, dtype=torch.long)

        outputs_start1, outputs_end1 = model1(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )
        
        outputs_start2, outputs_end2 = model2(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )
        
        outputs_start3, outputs_end3 = model3(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )
        
        outputs_start4, outputs_end4 = model4(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )
        
        outputs_start5, outputs_end5 = model5(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )
        outputs_start = (outputs_start1 + outputs_start2 + outputs_start3 + outputs_start4 + outputs_start5) / 5
        outputs_end = (outputs_end1 + outputs_end2 + outputs_end3 + outputs_end4 + outputs_end5) / 5
        
        outputs_start = torch.softmax(outputs_start, dim=1).cpu().detach().numpy()
        outputs_end = torch.softmax(outputs_end, dim=1).cpu().detach().numpy()
        jaccard_scores = []
        for px, tweet in enumerate(orig_tweet):
            selected_tweet = orig_selected[px]
            tweet_sentiment = sentiment[px]
            _, output_sentence = calculate_jaccard_score(
                original_tweet=tweet,
                target_string=selected_tweet,
                sentiment_val=tweet_sentiment,
                idx_start=np.argmax(outputs_start[px, :]),
                idx_end=np.argmax(outputs_end[px, :]),
                offsets=offsets[px]
            )
            
            output_sentence = pp(output_sentence, tweet)
            if output_sentence is None or pd.isna(output_sentence):
                output_sentence = tweet

            output_sentence = " ".join(str(output_sentence).split())
            if output_sentence == "":
                output_sentence = " ".join(str(tweet).split())

            final_output.append(output_sentence)

In [ ]:
sample = pd.read_csv("../input/tweet-sentiment-extraction/sample_submission.csv")

if len(final_output) != len(sample):
    raise ValueError(f"Prediction count mismatch: {len(final_output)} predictions for {len(sample)} rows")

submission = sample.copy()
submission["selected_text"] = pd.Series(final_output, index=submission.index).fillna("").astype(str)

empty_mask = submission["selected_text"].str.strip().eq("")
submission.loc[empty_mask, "selected_text"] = df_test.loc[empty_mask, "text"].astype(str)

submission["selected_text"] = submission["selected_text"].fillna("").astype(str)
assert not submission["selected_text"].isnull().any(), "selected_text contains null values"

submission.to_csv("submission.csv", index=False)
submission.head()


In [ ]:
sample.head()